# preprocessing mobile phone recording
        enregistrement smartphone
                ↓
        détection du jet
                ↓
        correction AGC
                ↓
        normalisation multi-appareils (robustesse téléphone)
                ↓
        extraction features librosa
                ↓
        modèle ML débit


Pipeline complet recommandé pour ton projet:

        file1 = preprocess_uroflow_audio("raw.wav", "step1.wav")

        file2 = device_normalization_wav(file1, "step2.wav")

        audio, sr = librosa.load(file2, sr=None)

        features = extract_features(audio, sr)

In [ ]:
import numpy as np
import librosa
import soundfile as sf

# normalisation basée sur niveau avant AGC, seule la la zone après AGC est réalignée correctement


def preprocess_uroflow_calibration(
    input_wav,
    output_wav,
    target_sr=None,
    frame_length=2048, # 2048 pour mieux détecter les changements de niveau liés à l'AGC, 1024 peut être trop court et plus sensible au bruit
    hop_length=512, # 512 pour une meilleure résolution temporelle dans la détection de l'AGC, 256 peut être trop court et plus sensible au bruit
    agc_sensitivity=2.5, # plus élevé = plus sensible à l'AGC, 2.5 est un bon point de départ pour les enregistrements téléphoniques
    agc_gain_limit=1.2 # Avec 2 le son est bq plus fort que l'original
):
    """
    Preprocessing calibration avec correction AGC
    sans augmenter la partie avant la réduction du téléphone.
    """

    # -------------------------
    # 1. Chargement audio
    # -------------------------
    y, sr = librosa.load(input_wav, sr=target_sr, mono=True)

    # -------------------------
    # 2. Analyse RMS
    # -------------------------
    rms = librosa.feature.rms(
        y=y,
        frame_length=frame_length,
        hop_length=hop_length
    )[0]

    diff = np.diff(rms)

    agc_threshold = -np.std(diff) * agc_sensitivity
    agc_candidates = np.where(diff < agc_threshold)[0]

    agc_frame = None

    for c in agc_candidates:
        if c > 10 and c + 10 < len(rms):
            pre = np.mean(rms[c-10:c])
            post = np.mean(rms[c:c+10])

            if post < pre * 0.6:
                agc_frame = c
                break

    # -------------------------
    # 3. Correction AGC
    # -------------------------
    if agc_frame is not None:
        agc_sample = agc_frame * hop_length

        pre_rms = np.mean(rms[max(0, agc_frame - 20):agc_frame])
        post_rms = np.mean(rms[agc_frame:agc_frame + 20])

        if post_rms > 0:
            gain = pre_rms / post_rms
            gain = min(gain, agc_gain_limit)

            y[agc_sample:] = y[agc_sample:] * gain

        # référence pour normalisation = niveau AVANT AGC
        ref_segment = y[:agc_sample]

    else:
        # si pas d'AGC détecté
        ref_segment = y

    # -------------------------
    # 4. Normalisation basée sur la référence stable
    # -------------------------
    rms_ref = np.sqrt(np.mean(ref_segment**2)) + 1e-8
    y = y / rms_ref

    percentile_95 = np.percentile(np.abs(ref_segment), 95)
    y = y / (percentile_95 + 1e-8)

    target_level = 1  # 0.0 à 1.0 # pour réduire l'amplitude sonore du jet et éviter les saturations
    y = y * target_level
    y = np.clip(y, -1, 1)

    # y = librosa.util.normalize(y) # sert à quelque chose?
    # y*= 0.7 # pour réduire l'amplitude sonore du jet et éviter les saturations

    # -------------------------
    # 5. Export WAV
    # -------------------------
    sf.write(output_wav, y, sr)

    print("Calibration audio preprocess terminé")
    print("AGC détecté :", agc_frame is not None)
    print("Durée :", len(y) / sr, "secondes")

    return output_wav

In [59]:
preprocess_uroflow_calibration(
    "/home/ed_st/code/anteecip/app_streamlit/12-Projet_miction/20_fev/A51/15.331_mls_A51.wav",
    "/home/ed_st/code/anteecip/app_streamlit/12-Projet_miction/20_fev/A51/15.331_mls_A51.wav_corrected.wav"
)

Calibration audio preprocess terminé
AGC détecté : True
Durée : 3.25 secondes


'/home/ed_st/code/anteecip/app_streamlit/12-Projet_miction/20_fev/A51/15.331_mls_A51.wav_corrected.wav'

# Version robuste et agnostique aux téléphones

In [ ]:
import numpy as np
import librosa
import soundfile as sf


def correct_agc_robust(
    input_path,
    output_path,
    frame_length=2048,
    hop_length=512,
    agc_sensitivity=2.5,
    agc_gain_limit=4.0,
    min_agc_duration=0.5
):
    """
    Correction AGC robuste multi-smartphones.

    Objectifs :
    - Ne jamais modifier le signal avant AGC
    - Détecter les AGC brutaux OU progressifs
    - Corriger uniquement la partie compressée
    - Ramener l'amplitude au niveau du jet réel

    Paramètres
    ----------
    input_path : str
        Chemin du fichier WAV brut.

    output_path : str
        Chemin du fichier WAV corrigé.

    frame_length : int
        Taille de la fenêtre d'analyse RMS.

    hop_length : int
        Pas entre les fenêtres RMS.

    agc_sensitivity : float
        Sensibilité de détection AGC.
        2.0 = sensible
        3.0 = strict

    agc_gain_limit : float
        Gain maximum appliqué pour éviter les erreurs.

    min_agc_duration : float
        Durée minimale (secondes) pour confirmer un AGC réel.
        Évite de corriger des variations naturelles du flux.
    """

    y, sr = librosa.load(input_path, sr=None, mono=True)

    # -------------------------
    # Analyse RMS
    # -------------------------
    rms = librosa.feature.rms(
        y=y,
        frame_length=frame_length,
        hop_length=hop_length
    )[0]

    diff = np.diff(rms)

    threshold = -np.std(diff) * agc_sensitivity
    candidates = np.where(diff < threshold)[0]

    agc_frame = None

    min_frames = int((min_agc_duration * sr) / hop_length)

    # -------------------------
    # Détection robuste AGC
    # -------------------------
    for c in candidates:

        if c > 20 and c + min_frames < len(rms):

            pre = np.median(rms[c-20:c])
            post = np.median(rms[c:c+min_frames])

            drop_ratio = post / (pre + 1e-8)

            # chute forte ou compression progressive
            if drop_ratio < 0.65:
                agc_frame = c
                break

    # -------------------------
    # Correction AGC
    # -------------------------
    if agc_frame is not None:

        agc_sample = agc_frame * hop_length

        # zone stable avant AGC
        ref_start = int(max(0, agc_sample - sr * 1.5))
        ref_end = agc_sample

        ref_segment = y[ref_start:ref_end]

        rms_ref = np.sqrt(np.mean(ref_segment**2)) + 1e-8
        rms_post = np.sqrt(np.mean(y[agc_sample:]**2)) + 1e-8

        gain = rms_ref / rms_post
        gain = min(gain, agc_gain_limit)

        y_corrected = y.copy()

        # correction uniquement après AGC
        y_corrected[agc_sample:] *= gain

    else:
        y_corrected = y

    # sécurité clipping
    y_corrected = np.clip(y_corrected, -1, 1)

    sf.write(output_path, y_corrected, sr)

    print("AGC détecté :", agc_frame is not None)
    if agc_frame is not None:
        print("Temps AGC :", round(agc_frame * hop_length / sr, 2), "s")

    return output_path

# Fonction : normalisation acoustique multi-téléphones

In [41]:
import numpy as np
import librosa
import soundfile as sf


def device_normalization_wav(
    input_wav,
    output_wav="normalized.wav",
    n_fft=2048,
    hop_length=512,
    strength=0.8
):
    """
    Réduit la signature acoustique spécifique au téléphone
    tout en conservant les caractéristiques utiles pour le ML.
    """

    audio, sr = librosa.load(input_wav, sr=None, mono=True)

    # STFT
    S_complex = librosa.stft(audio, n_fft=n_fft, hop_length=hop_length)
    S_mag = np.abs(S_complex)
    phase = np.angle(S_complex)

    # profil spectral moyen du téléphone
    spectral_profile = np.mean(S_mag, axis=1, keepdims=True)

    spectral_profile += 1e-8

    # normalisation partielle (évite d'écraser l'information utile)
    S_mag_norm = S_mag / (spectral_profile ** strength)

    # reconstruction
    S_new = S_mag_norm * np.exp(1j * phase)
    audio_norm = librosa.istft(S_new, hop_length=hop_length)

    # normalisation amplitude finale
    audio_norm = audio_norm / np.max(np.abs(audio_norm)) * 0.9

    sf.write(output_wav, audio_norm, sr)

    return output_wav

# pipeline complet pour la mesure du débit d'un fluex réel de miction à partir d'un enregistrement smartphone:


        téléphone → enregistrement
                ↓
        preprocess_uroflow_audio ("raw.wav", "step1.wav") --> correction AGC + détection du jet ("step2.wav")
                ↓
        wav nettoyé
                ↓
        features 0.2 s
                ↓
        modèle débit
                ↓
        courbe uroflow

In [42]:
import numpy as np
import librosa
import soundfile as sf
'''
Principes utilisés dans ce code :

    Détection automatique du jet par analyse RMS dynamique

    Découpe intelligente (pas basée sur 0,2 s mais sur le signal réel)

    Correction AGC (détection chute d’énergie progressive)

    Normalisation multi-téléphones

    Préservation des variations rapides (important pour tes fenêtres 0,2 s)

    Export d’un nouveau WAV prêt pour ton extraction de features

La découpe ne doit pas être faite sur 0,2 s ici — c’est mieux de le faire après
'''



def preprocess_uroflow_audio(
    input_wav,                # chemin complet du fichier .wav d'entrée
    output_wav,               # chemin du nouveau fichier .wav exporté
    target_sr=None,           # fréquence d'échantillonnage cible (uniformise tous les téléphones)# 22050 fréquence cible pour homogénéiser les enregistrements. None = garde la fréquence native du téléphone en général 16kHz. 
    frame_length=2048,        # taille de fenêtre pour analyse RMS (stabilité énergie)
    hop_length=512,           # pas entre fenêtres RMS
    max_internal_silence=0.5, # durée max silence interne conservé (secondes)
    top_db=30,                # seuil de suppression silence début/fin
    agc_sensitivity=2.5,      # sensibilité détection AGC (plus grand = détection plus stricte)
    agc_gain_limit=2.0        # gain maximum autorisé lors de la correction AGC
):
    """
    Préprocessing audio pour uroflow acoustique.

    Pipeline :
    1. Chargement audio
    2. Suppression silence début/fin
    3. Détection automatique du jet
    4. Détection AGC smartphone
    5. Correction AGC immédiate mais limitée
    6. Normalisation multi-device
    7. Gestion des silences internes (sans supprimer les zones AGC)
    8. Export WAV final
    """

    # -------------------------
    # 1. Chargement audio
    # -------------------------
    y, sr = librosa.load(input_wav, sr=target_sr, mono=True)

    # -------------------------
    # 2. Suppression silence début/fin
    # -------------------------
    y_trimmed, _ = librosa.effects.trim(y, top_db=top_db)

    # -------------------------
    # 3. Détection du jet
    # -------------------------
    rms = librosa.feature.rms(
        y=y_trimmed,
        frame_length=frame_length,
        hop_length=hop_length
    )[0]

    # estimation bruit de fond
    noise_floor = np.percentile(rms, 20)

    # seuil jet
    jet_threshold = noise_floor * 3

    jet_frames = np.where(rms > jet_threshold)[0]

    if len(jet_frames) == 0:
        print("Jet non détecté")
        return None

    start_frame = jet_frames[0]
    end_frame = jet_frames[-1]

    start_sample = start_frame * hop_length
    end_sample = end_frame * hop_length

    y = y_trimmed[start_sample:end_sample]

    # -------------------------
    # 4. Détection AGC smartphone
    # -------------------------
    rms2 = librosa.feature.rms(
        y=y,
        frame_length=frame_length,
        hop_length=hop_length
    )[0]

    diff = np.diff(rms2)

    # seuil AGC basé sur variabilité du signal
    agc_threshold = -np.std(diff) * agc_sensitivity

    agc_candidates = np.where(diff < agc_threshold)[0]

    agc_frame = None

    # vérifier que la chute est réelle et durable
    for c in agc_candidates:
        if c > 10 and c + 10 < len(rms2):
            pre = np.mean(rms2[c-10:c])
            post = np.mean(rms2[c:c+10])

            # vraie chute AGC
            if post < pre * 0.6:
                agc_frame = c
                break

    # -------------------------
    # 5. Correction AGC immédiate
    # -------------------------
    if agc_frame is not None:
        agc_sample = agc_frame * hop_length

        pre_rms = np.mean(rms2[max(0, agc_frame - 20):agc_frame])
        post_rms = np.mean(rms2[agc_frame:agc_frame + 20])

        if post_rms > 0:
            gain = pre_rms / post_rms
            gain = min(gain, agc_gain_limit)

            y[agc_sample:] = y[agc_sample:] * gain

    # -------------------------
    # 6. Normalisation multi-device
    # -------------------------
    rms_global = np.sqrt(np.mean(y**2)) + 1e-8
    y = y / rms_global

    percentile_95 = np.percentile(np.abs(y), 95)
    y = y / (percentile_95 + 1e-8)
    target_level = 0.9  # 0.0 à 1.0 # pour réduire l'amplitude sonore du jet et éviter les saturations
    y = y * target_level
    y = np.clip(y, -1, 1)

    # -------------------------
    # 7. Gestion silences internes
    # IMPORTANT : ne pas confondre AGC avec silence
    # -------------------------
    silence_threshold = noise_floor * 1.2

    rms3 = librosa.feature.rms(
        y=y,
        frame_length=frame_length,
        hop_length=hop_length
    )[0]

    max_silence_frames = int((max_internal_silence * sr) / hop_length)

    mask = rms3 > silence_threshold

    silence_len = 0
    new_mask = np.zeros_like(mask)

    for i, m in enumerate(mask):
        if m:
            silence_len = 0
            new_mask[i] = 1
        else:
            silence_len += 1
            if silence_len <= max_silence_frames:
                new_mask[i] = 1
            else:
                new_mask[i] = 0

    # reconstruction signal
    y_frames = librosa.util.frame(
        y,
        frame_length=hop_length,
        hop_length=hop_length
    ).T

    y_filtered = y_frames[new_mask].flatten()

    # -------------------------
    # 8. Export final
    # -------------------------
    sf.write(output_wav, y_filtered, sr)

    print("Audio preprocess terminé")
    print("AGC détecté :", agc_frame is not None)
    print("Durée finale :", len(y_filtered) / sr, "secondes")

    return output_wav